## nb28 — Author Paper Panel Collection

For each author (603 award + 603 control from matched_pairs_clean.csv),
fetch all their papers in the window [award_year-5, award_year+5] from OpenAlex.

Output: `data/cd_trajectory/author_papers_panel.csv`
Columns: author_id, paper_id, paper_year, award_year, group, relative_year

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path

ROOT    = Path('..')
MATCHED = ROOT / 'data' / 'matched'
OUT_DIR = ROOT / 'data' / 'cd_trajectory'
OUT_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = 'A08hCjeUoeVKA9toVsfCpF'
WINDOW  = 5

In [ ]:
pairs = pd.read_csv(MATCHED / 'matched_pairs_clean.csv')
print(pairs.shape)
print(pairs.columns.tolist())
pairs.head(3)

In [ ]:
treated_list = pairs[['treated_id', 'award_year']].rename(columns={'treated_id': 'author_id'})
treated_list['group'] = 'award'

control_list = pairs[['control_id', 'award_year']].rename(columns={'control_id': 'author_id'})
control_list['group'] = 'control'

authors = pd.concat([treated_list, control_list], ignore_index=True)
authors = authors.drop_duplicates(subset='author_id').reset_index(drop=True)

print(f'Total unique authors: {len(authors)}')
print(authors['group'].value_counts())

In [ ]:
def fetch_author_papers(author_id, year_min, year_max):
    aid = author_id.split('/')[-1]
    url = 'https://api.openalex.org/works'
    papers = []
    cursor = '*'
    while True:
        params = {
            'filter': f'author.id:{aid},publication_year:{year_min}-{year_max}',
            'per-page': 200,
            'select': 'id,publication_year',
            'cursor': cursor,
            'api_key': API_KEY,
        }
        r = requests.get(url, params=params, timeout=15)
        if r.status_code != 200:
            break
        data = r.json()
        results = data.get('results', [])
        for w in results:
            papers.append({'paper_id': w['id'].split('/')[-1], 'paper_year': w['publication_year']})
        cursor = data.get('meta', {}).get('next_cursor')
        if not cursor or not results:
            break
        time.sleep(0.15)
    return papers

In [ ]:
CHECKPOINT = OUT_DIR / 'author_papers_panel.csv'

if CHECKPOINT.exists():
    done_df  = pd.read_csv(CHECKPOINT)
    done_ids = set(done_df['author_id'].unique())
    print(f'Resuming — {len(done_ids)} authors already done')
else:
    done_df  = pd.DataFrame()
    done_ids = set()

rows = []
todo = authors[~authors['author_id'].isin(done_ids)].reset_index(drop=True)
print(f'Remaining: {len(todo)} authors')

for i, row in todo.iterrows():
    aid        = row['author_id']
    award_year = int(row['award_year'])
    group      = row['group']
    yr_min     = award_year - WINDOW
    yr_max     = award_year + WINDOW

    papers = fetch_author_papers(aid, yr_min, yr_max)
    for p in papers:
        rows.append({
            'author_id':     aid,
            'group':         group,
            'award_year':    award_year,
            'paper_id':      p['paper_id'],
            'paper_year':    p['paper_year'],
            'relative_year': p['paper_year'] - award_year,
        })

    if (i + 1) % 50 == 0 or (i + 1) == len(todo):
        batch_df = pd.DataFrame(rows)
        combined = pd.concat([done_df, batch_df], ignore_index=True)
        combined.to_csv(CHECKPOINT, index=False)
        done_df  = combined
        rows     = []
        print(f'[{i+1}/{len(todo)}] Saved — total rows: {len(done_df)}')

    time.sleep(0.2)

print('Done!')
print(done_df.groupby('group')['author_id'].nunique())

In [ ]:
panel = pd.read_csv(CHECKPOINT)
print(panel.shape)
print(panel.groupby('group')['author_id'].nunique())
print(panel.groupby('group')['paper_id'].nunique())
panel.head()